In [1]:
from prob_modeling import *
import warnings

In [2]:
def train_model(df, mean_model_names, resid_model_names, need_prep=True, use_hier_feats_mean=True, use_hier_feats_resid=True):
    if use_hier_feats_mean:
        hier_mean_feats_mean = HierMeanFeatureSet().get_feature_names_out()
    else:
        hier_mean_feats_mean = []

    if use_hier_feats_resid:
        hier_mean_feats_resid = HierMeanFeatureSet().get_feature_names_out()
    else:
        hier_mean_feats_resid = []

    preprocessor_mean = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
            ("num", "passthrough", NUM_FEATURES + hier_mean_feats_mean),
        ]
    )

    preprocessor_resid = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
            ("num", "passthrough", NUM_FEATURES + hier_mean_feats_resid + ["y_pred"]),
        ]
    )

    name_to_params_map = {  
        "HM": {},
        "LR": LR_PARAM_GRID,
        "RF": RF_PARAM_GRID,
        "XGB": XGB_PARAM_GRID,
        "LGBM": LGBM_PARAM_GRID
    }

    name_to_class_map_mean = {
        "HM": HierMean,
        "LR": get_pipeline_wrap(preprocessor_mean, LinearRegression),
        "RF": get_pipeline_wrap(preprocessor_mean, RandomForestRegressor),
        "XGB": get_pipeline_wrap(preprocessor_mean, XGBRegressor),
        "LGBM": get_pipeline_wrap(preprocessor_mean, LGBMRegressor)
    }

    name_to_class_map_resid = {
        "HM": HierMean,
        "LR": get_pipeline_wrap(preprocessor_resid, LinearRegression),
        "RF": get_pipeline_wrap(preprocessor_resid, RandomForestRegressor),
        "XGB": get_pipeline_wrap(preprocessor_resid, XGBRegressor),
        "LGBM": get_pipeline_wrap(preprocessor_resid, LGBMRegressor)
    }

    if need_prep:
        df = prep_df(df)

    train_df, val_df, test_df = get_train_val_test_split(df)
    best_model, mean_model_summary, full_model_summary = predict_winning_price_scale_model(
        train_df, val_df, test_df,
        features=FEATURES+HierMeanFeatureSet().get_feature_names_out()+[TARGET], # target needed for HM model fitting, not used in prediction & and in other models 
        target=TARGET,
        mean_model_names=mean_model_names,
        resid_model_names=resid_model_names,
        mean_model_classes=[name_to_class_map_mean[n] for n in mean_model_names],
        resid_model_classes=[name_to_class_map_resid[n] for n in resid_model_names],
        param_grid_map=name_to_params_map,
        use_y_pred=True
    )
    return best_model, mean_model_summary, full_model_summary


In [ ]:
warnings.filterwarnings(
    "ignore",
    message="The total space of parameters .* is smaller than n_iter"
)

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

from itertools import product
from query_logic import query

query_map = {
    "hiv": "product: (rapid OR nhanh) AND hiv",
    "hcv": "product: (rapid OR nhanh) AND hcv",
    "hbv": "product: (rapid OR nhanh) AND (hbv OR hbsag)"
}
mean_models = ["HM", "LR", "RF", "XGB", "LGBM"]
resid_models = ["LR", "RF", "XGB", "LGBM"]
combos = list(product(mean_models, resid_models))
all_res = []
for ds in ["hiv", "hbv", "hcv"]:
    df = query(query_map[ds])
    df["closing_date"] = pd.to_datetime(df["closing_date"], dayfirst=False)
    print(len(df))
    for mean_model, resid_model in combos:
        model_summary = {"ds": ds}
        best_model, mean_model_summary, full_model_summary = train_model(df, [mean_model], [resid_model])
        model_summary.update(best_model["metrics"])
        all_res.append(model_summary)

In [5]:
pd.DataFrame(all_res).to_csv("benchmark_res.csv", index=False)